# Tech Addiction Prediction: Pseudo-Labeling with XGBoost
In this notebook, we implement **Pseudo-Labeling**, a powerful semi-supervised learning technique.
1. **Train** a strong model on the labeled Train set.
2. **Predict** on the unlabeled Test set.
3. **Filter** the highly confident predictions (e.g., > 0.95 or < 0.05) and assign them hard labels (1 and 0).
4. **Combine** these highly confident Test samples with the original Train set.
5. **Retrain** the model on this expanded dataset to improve generalization.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')


## 1. Data Loading & Feature Engineering (V5)


In [ ]:
TRAIN_PATH = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
TEST_PATH = '/kaggle/input/competitions/playground-series-s6e8/test.csv'

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

def engineer_features(train, test):
    train = train.copy()
    test = test.copy()
    for df in [train, test]:
        df['weekend_delta'] = df['weekend_screen_time'] - df['daily_screen_time_hours']
        df['social_media_prop'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['unaccounted_screen_time'] = df['daily_screen_time_hours'] - (df['social_media_hours'] + df['gaming_hours'] + df['work_study_hours'])
        df['productivity_ratio'] = df['work_study_hours'] / (df['daily_screen_time_hours'] + 1e-5)
        df['entertainment_ratio'] = (df['social_media_hours'] + df['gaming_hours']) / (df['daily_screen_time_hours'] + 1e-5)
        df['screen_to_sleep_ratio'] = df['daily_screen_time_hours'] / (df['sleep_hours'] + 1e-5)
        df['interaction_intensity'] = df['notifications_per_day'] * df['app_opens_per_day']
        df['sleep_deprived'] = (df['sleep_hours'] < 6.5).astype(int)
        df['age_group'] = pd.cut(df['age'], bins=[0, 20, 30, 40, 50, 100], labels=False)
    return train, test

X = train_df.drop(['id', 'addicted_label'], axis=1)
y = train_df['addicted_label'].values
X_test_orig = test_df.drop(['id'], axis=1)

X, X_test = engineer_features(X, X_test_orig)

categorical_features = ['gender', 'academic_work_impact', 'stress_level', 'age_group']
for col in categorical_features:
    X[col] = X[col].astype(str).astype('category')
    X_test[col] = X_test[col].astype(str).astype('category')


## 2. Train Initial Model & Generate Pseudo-Labels


In [ ]:
# Using hyper-parameters typically found via Optuna for XGBoost V5
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'random_state': 42,
    'tree_method': 'hist',
    'learning_rate': 0.02,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_estimators': 1500,
    'enable_categorical': True
}

print("Training initial model on labeled data...")
initial_model = xgb.XGBClassifier(**xgb_params)
initial_model.fit(X, y)

print("Predicting on Test set to generate probabilities...")
test_probs = initial_model.predict_proba(X_test)[:, 1]

CONFIDENCE_THRESHOLD_HIGH = 0.95
CONFIDENCE_THRESHOLD_LOW = 0.05

pseudo_idx = np.where((test_probs > CONFIDENCE_THRESHOLD_HIGH) | (test_probs < CONFIDENCE_THRESHOLD_LOW))[0]
print(f"Found {len(pseudo_idx)} highly confident samples out of {len(X_test)}")

X_pseudo = X_test.iloc[pseudo_idx].copy()
y_pseudo = (test_probs[pseudo_idx] > 0.5).astype(int)


## 3. Combine Data and Retrain with 5-Fold CV


In [ ]:
X_combined = pd.concat([X, X_pseudo]).reset_index(drop=True)
y_combined = np.concatenate([y, y_pseudo])

print(f"New combined training set size: {len(X_combined)}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_combined, y_combined)):
    print(f"\n--- Training Fold {fold + 1}/5 ---")
    X_train_f, X_val_f = X_combined.iloc[train_idx], X_combined.iloc[val_idx]
    y_train_f, y_val_f = y_combined[train_idx], y_combined[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=100)
    model.fit(
        X_train_f, y_train_f,
        eval_set=[(X_val_f, y_val_f)],
        verbose=False
    )
    
    test_preds += model.predict_proba(X_test)[:, 1] / 5

print("\nRetraining complete!")


## 4. Submission


In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'addicted_label': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission saved!")
display(submission.head())
